In [ ]:
!pip install psycopg-binary psycopg psycopg_c asyncio
!pip install folium
!pip install geopandas
!pip install shapely

fish: Unknown command: q
~/.config/fish/config.fish (line 21): 
q
^
from sourcing file ~/.config/fish/config.fish
	called during startup

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
fish: Unknown command: q
~/.config/fish/config.fish (line 21): 
q
^
from sourcing file ~/.config/fish/config.fish
	called during startup

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
fish: Unknown command: q
~/.config/fish/config.fish (line 21): 
q
^
from sourcing file ~/.config/fish/config.fish
	called during startup

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
fish: Unknown command: q
~/.config/fish/config.fish (line 21): 
q
^
from sourcing file ~/.config/fish/config.fish
	called during startup

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
import urllib3.util.connection as urllib3_conn
urllib3_conn.HAS_IPV6 = False
import pandas as pd
import socket
import numpy as np
import requests
import time
from pathlib import Path
from tqdm import tqdm
import os
from pathlib import Path
import json
import asyncio
import joblib
import folium
import geopandas as gdf
from shapely.geometry import mapping

In [ ]:
print(os.getcwd())
project_root = Path.cwd().parent
damn_data = project_root/"src"/"resources"/"all_dams_weekly.csv"
dam_location_data = project_root/"src"/"resources"/"african_dams-African Dams.csv"
feaature_data_csv = project_root/"data"/"feature_engineered"/"engineered_data.csv"
south_africa_border_json = project_root/"src"/"resources"/"South_africa_version_0_geodata.json"
model_30 = joblib.load(project_root/"models"/"xgboost_rainfall_30.pkl")
model_60 = joblib.load(project_root/"models"/"xgboost_rainfall_60.pkl")

/home/chiyedza/Downloads/Personal projects/SA-Water-Dam-Level-Predictor/notebooks


In [ ]:
dam_df = pd.read_csv(damn_data)
# dam_df.head(20)

In [ ]:
south_africa_border_geodata = gdf.read_file(south_africa_border_json)
south_africa_border_geodata = south_africa_border_geodata.to_crs(epsg=4326)
south_africa_border_geodata = south_africa_border_geodata.__geo_interface__


In [ ]:
dam_location_df = pd.read_csv(dam_location_data)
dam_location_SA = dam_location_df[dam_location_df["Country"] == "South Africa"]
# dam_location_SA["Name of dam"].head(50)

In [ ]:
complete_featured_data = pd.read_csv(feaature_data_csv)
featured_data_df = pd.read_csv(feaature_data_csv)
featured_data_30_df = featured_data_df.drop(["date", "lag_1_target_30","target_30"],axis=1)
featured_data_60_df = featured_data_df.drop(["date", "lag_1_target_30","target_60"],axis=1)
complete_featured_data.head()

,date,water_level_mm,year,month,lag_1,lag_7,lag_14,lag_28,lag_56,lag_84,...,days_since_significant_rain,quarter,wet_season,wet_dry_flag,season_start_year,days_into_season,target_30,target_60,target_90,lag_1_target_30
0,2000-03-25,2.667252,2000,3,7.038533,5.152590,0.866577,0.713604,1.256391,9.107801,...,0,1,3.0,True,1999,84,47.778553,64.367526,72.975637,50.116940
1,2000-03-26,5.065514,2000,3,2.667252,1.081687,1.942799,0.627804,0.068120,5.781268,...,0,1,3.0,True,1999,85,43.151141,60.459946,67.910140,47.778553
2,2000-03-27,3.869131,2000,3,5.065514,3.774888,1.212185,0.799240,0.054767,1.829304,...,0,1,3.0,True,1999,86,39.389662,57.071526,64.042135,43.151141
3,2000-03-28,2.482193,2000,3,3.869131,1.087285,1.506839,1.622073,0.119573,2.341147,...,0,1,3.0,True,1999,87,37.156184,54.590095,61.559941,39.389662
4,2000-03-29,3.010641,2000,3,2.482193,1.344326,2.519668,3.079031,0.881478,5.645532,...,0,1,3.0,True,1999,88,34.791240,51.752334,58.549301,37.156184


In [ ]:
filtered_dam_df = dam_df[dam_df["Dam"] != "Total"]
# filtered_dam_df.info()

In [ ]:
clean_name_lst = []
dam_names = filtered_dam_df["Dam"]
print(len(dam_names))

for vals in dam_names:
    name_lst = vals.split(" ")
    clean_name_lst.append(name_lst[0])
print(len(clean_name_lst))

filtered_dam_df["Clean_names"] = clean_name_lst
print(clean_name_lst)


178
178
['Bon', 'Bronkhorstspruit', 'Klipdrift', 'Rietvlei', 'Roodeplaat', 'Katse', 'Mohale', 'Blyderivierpoort', 'Buffelskloof', 'Da', 'Driekoppies', 'Grootdraai', 'Heyshope', 'Inyaka', 'Jericho', 'Klipkopjes', 'Kwena', 'Longmere', 'Loskop', 'Middelburg', 'Morgenstond', 'Nooitgedacht', 'Ohrigstad', 'Primkop', 'Rhenosterkop', 'Vygeboom', 'Westoe', 'Witbank', 'Witklip', 'Boskop', 'Bospoort', 'Buffelspoort', 'Disaneng', 'Elandskuil', 'Hartbeespoort', 'Johan', 'Klein-Maricopoort', 'Klerkskraal', 'Klipvoor', 'Kosterrivier', 'Kromellenboog', 'Lindleyspoort', 'Madikwe', 'Marico-Bosveld', 'Middelkraal', 'Molatedi', 'Ngotwane', 'Olifantsnek', 'Pella', 'Potchefstroom', 'Rietspruit', 'Roodekopjes', 'Sehujwane', 'Setumo', 'Swartruggens', 'Taung', 'Vaalkop', 'Allemanskraal', 'Armenia', 'Bloemhof', 'Egmont', 'Erfenis', 'Fika-Patso', 'Gariep', 'Groothoek', 'Kalkfontein', 'Knellpoort', 'Koppies', 'Krugersdrift', 'Metsi-Matsho', 'Rustfontein', 'Saulspoort', 'Sterkfontein', 'Tierpoort', 'Vaal', 'Vaalri

In [ ]:
filtered_dam_df.head()

,Dam,River,Photo,Indicators,FSC,This Week,Last Week,Last Year,Clean_names
0,Bon Accord Dam,Apies River,NaN,Indicators,4.4,#103.0,#103.0,108.2,Bon
1,Bronkhorstspruit Dam,Bronkhorstspruit River,Photo,Indicators,57.0,98.6,98.9,100.4,Bronkhorstspruit
2,Klipdrift Dam,Loopspruit River,Photo,Indicators,13.4,99.7,#102.2,97.7,Klipdrift
3,Rietvlei Dam,Hennops River,Photo,Indicators,12.3,98.9,92.7,100.2,Rietvlei
4,Roodeplaat Dam,Pienaars River,Photo,Indicators,41.2,98.3,99.7,100.5,Roodeplaat


In [ ]:
active_dams  = dam_location_SA[dam_location_SA["Name of dam"].isin(filtered_dam_df["Clean_names"])]
filtered_active_dams = filtered_dam_df[filtered_dam_df["Clean_names"].isin(active_dams["Name of dam"])]
filtered_active_dams.head()

lat_map = active_dams.set_index("Name of dam")["Decimal degree latitude"]
long_map = active_dams.set_index("Name of dam")["Decimal degree longitude"]

filtered_active_dams["lat"] = filtered_active_dams["Clean_names"].map(lat_map)
filtered_active_dams["long"] = filtered_active_dams["Clean_names"].map(long_map)

filtered_active_dams.head()

,Dam,River,Photo,Indicators,FSC,This Week,Last Week,Last Year,Clean_names,lat,long
1,Bronkhorstspruit Dam,Bronkhorstspruit River,Photo,Indicators,57.0,98.6,98.9,100.4,Bronkhorstspruit,-25.887,28.721
4,Roodeplaat Dam,Pienaars River,Photo,Indicators,41.2,98.3,99.7,100.5,Roodeplaat,-25.621,28.373
10,Buffelskloof Dam,Waterval River,Photo,Indicators,5.3,95.3,97.8,91.0,Buffelskloof,-24.956,30.265
13,Grootdraai Dam,Vaal River,Photo,Indicators,349.8,99.2,100.0,92.7,Grootdraai,-29.300,26.917
14,Heyshope Dam,Assegaai River,Photo,Indicators,445.0,99.8,100.0,100.0,Heyshope,-27.000,30.167


In [ ]:
filtered_active_dams["lat"] = pd.to_numeric(filtered_active_dams["lat"], errors="coerce")
filtered_active_dams["long"] = pd.to_numeric(filtered_active_dams["long"], errors="coerce")


In [ ]:
day_30_pred = model_30.predict(featured_data_30_df)[-1]
day_60_pred = model_60.predict(featured_data_60_df)[-1]


In [ ]:
"""Function that converts 30 day prediction to visual colour"""
def mm_to_blue(mm, vmin=0, vmax=80):
    mm = max(vmin, min(vmax,mm))
    t = (mm - vmin) / (vmax - vmin)
    r = int(204 + t * (51-204))
    g = int(230 + t * (153-230))
    b = 255
    return f"#{r:02x}{g:02x}{b:02x}"

predict_30_date = complete_featured_data.iat[-1, 0]
print(predict_30_date)

tooltip_text = (
    f"30-day-rainfall forcast: {day_30_pred:,.1f} mm<br>" 
    f"Expected accumulation across SA<br>"
    f"Based on date {predict_30_date}"
)

fill_color = mm_to_blue(day_30_pred)

style_dict = {
    "fillColor":fill_color,
    "color":"#0066cc",
    "weight": 0.5,
    "fillOpacity":0.15
}

"""Converts 60 day predictions to visual colours"""
def mm_to_purple(mm, vmin=0, vmax=80):
    mm = max(vmin, min(vmax,mm))
    t = (mm - vmin) / (vmax - vmin)
    r = int(350 + t * (51-204))
    g = int(200+ t * (153-230))
    b = 280
    return f"#{r:02x}{g:02x}{b:02x}"

predict_date = complete_featured_data.iat[-1, 0]
print(predict_date)

tooltip_text_30 = (
    f"30-day-rainfall forcast: {day_30_pred:,.1f} mm<br>" 
    f"Expected accumulation across SA<br>"
    f"Based on date {predict_date}"
)

tooltip_text_60 = (
    f"60-day-rainfall forcast: {day_60_pred:,.1f} mm<br>" 
    f"Expected accumulation across SA<br>"
    f"Based on date {predict_date}"
)

fill_color_30 = mm_to_blue(day_30_pred)
fill_color_60 = mm_to_purple(day_60_pred)

style_dict_30 = {
    "fillColor":fill_color_30,
    "color":"#0066cc",
    "weight": 0.5,
    "fillOpacity":0.15
}

style_dict_60 = {
    "fillColor":fill_color_60,
    "color":"#0066cc",
    "weight": 0.5,
    "fillOpacity":0.15
}

2026-04-30
2026-04-30


In [ ]:
m = folium.Map(
    location = [-28.5, 24.6],
    tiles = "OpenStreetMap",
    zoom_start = 6
)

m_1 = folium.Map(
    location = [-28.5, 24.6],
    tiles = "OpenStreetMap",
    zoom_start = 6
)

colors = ["red", "yellow", "green"]
color = " "
for dams in range(0, len(filtered_active_dams)):
    if filtered_active_dams.iat[dams, 4] < 50:
        color = colors[0]
    elif filtered_active_dams.iat[dams, 4] >= 50 and filtered_active_dams.iat[dams, 4] <= 70:
        color = colors[1]
    else:
        color = colors[2]

    folium.Marker(
        location=[filtered_active_dams.iat[dams, 9], filtered_active_dams.iat[dams, 10]],
        popup = f"Dam Name:\n{filtered_active_dams.iat[dams, 0]}\nFSC:\n{filtered_active_dams.iat[dams,4]}",
        tooltip = "Click For Info",
        icon = folium.Icon(color=color, icon="cloud")
    ).add_to(m)

    folium.Marker(
        location=[filtered_active_dams.iat[dams, 9], filtered_active_dams.iat[dams, 10]],
        popup = f"Dam Name:\n{filtered_active_dams.iat[dams, 0]}\nFSC:\n{filtered_active_dams.iat[dams,4]}",
        tooltip = "Click For Info",
        icon = folium.Icon(color=color, icon="cloud")
    ).add_to(m_1)

tooltip_30 = folium.Tooltip(tooltip_text_30, sticky=False)
tooltip_60 = folium.Tooltip(tooltip_text_60, sticky=False)

folium.GeoJson(
    data = south_africa_border_geodata,
    name = "National Raainfaall Forcast",
    style_function = lambda x: style_dict_30,
    tooltip = tooltip_30,
).add_to(m)

folium.GeoJson(
    data = south_africa_border_geodata,
    name = "National Raainfaall Forcast",
    style_function = lambda x: style_dict_60,
    tooltip = tooltip_60,
).add_to(m_1)

m.save(project_root/"Graphs_and_Charts"/"folim_map_htmls"/"Dam_map_rain_prediction_day_30.html")
m_1.save(project_root/"Graphs_and_Charts"/"folim_map_htmls"/"Dam_map_rain_prediction_day_60.html")

/tmp/ipykernel_68764/2559950330.py:27: UserWarning: color argument of Icon should be one of: {'beige', 'gray', 'white', 'green', 'orange', 'pink', 'purple', 'black', 'red', 'darkblue', 'darkred', 'cadetblue', 'lightgray', 'lightgreen', 'blue', 'darkpurple', 'lightred', 'darkgreen', 'lightblue'}.
  icon = folium.Icon(color=color, icon="cloud")
/tmp/ipykernel_68764/2559950330.py:34: UserWarning: color argument of Icon should be one of: {'beige', 'gray', 'white', 'green', 'orange', 'pink', 'purple', 'black', 'red', 'darkblue', 'darkred', 'cadetblue', 'lightgray', 'lightgreen', 'blue', 'darkpurple', 'lightred', 'darkgreen', 'lightblue'}.
  icon = folium.Icon(color=color, icon="cloud")
